In [0]:
from pyspark.sql.functions import current_timestamp, col

# Ruta del archivo subido
file_path = "/Volumes/workspace/default/mining/MiningProcess_Flotation_Plant_Database.csv"

#Lectura del CSV en estado crudo (Bronze)
df_raw = (spark.read
          .option("header", "true")
          .option("inferSchema", "false") # En Bronze leemos sin forzar tipos para no perder datos
          .csv(file_path))

#Sanitizar nombres de columnas (Quitar % y espacios para cumplir reglas de Delta)
for column_name in df_raw.columns:
    clean_name = column_name.replace("%", "pct").replace(" ", "_").strip()
    df_raw = df_raw.withColumnRenamed(column_name, clean_name)

#Agregar metadatos de auditoría
df_bronze = df_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path"))


display(df_bronze.limit(5))

In [0]:
#Nombre completo de la tabla
table_name = "workspace.default.bronze_mining"

#Guardar el DataFrame como Tabla Delta en la Capa Bronze
(df_bronze.write
 .format("delta")
 .mode("overwrite")
 .option("mergeSchema", "true")
 .saveAsTable(table_name))

print(f"Tabla {table_name} guardada con éxito en la Capa Bronze.")